In [44]:
import pandas as pd
import csv
from dotenv import load_dotenv
from groq import Groq

**Pandas**  
When there is a CSV data, efficient and effective way to handle is to use pandas  
This can be used for various data handlind and manipulation  
That makes a good choice for Data Retirieval mechanism

In [45]:
Data = pd.read_csv ('Student_Performance.csv')
print (Data.dtypes)

student_id                           str
age                                int64
gender                               str
study_hours_per_day              float64
social_media_hours               float64
netflix_hours                    float64
part_time_job                        str
attendance_percentage            float64
sleep_hours                      float64
diet_quality                         str
exercise_frequency                 int64
parental_education_level             str
internet_quality                     str
mental_health_rating               int64
extracurricular_participation        str
exam_score                       float64
dtype: object


>Check pandas functions that can be used for data calculations and inference

In [46]:
# various data computation possible
print ('Number of records : ', len (Data))
print ('Min, Avg, Max Study Hours : ', Data['study_hours_per_day'].min(), Data['study_hours_per_day'].mean(), Data['study_hours_per_day'].max())
print ("Values in Paretal Education :", Data['parental_education_level'].unique())
print ("Values in Internet Quality :", Data['internet_quality'].unique())

Number of records :  1000
Min, Avg, Max Study Hours :  0.0 3.5501000000000005 8.3
Values in Paretal Education : <StringArray>
['Master', 'High School', 'Bachelor', nan]
Length: 4, dtype: str
Values in Internet Quality : <StringArray>
['Average', 'Poor', 'Good']
Length: 3, dtype: str


**Filtering**  
Pandas provides conditional filter / slicing functions that can be used as part of retrieval  
Result again in another pandas dataframe

In [47]:
# filter for poor internet quality and score more than 90
# All columns considered
Filtered = Data[(Data['internet_quality'] =='Poor') & (Data['exam_score'] > 90.0)]
Filtered

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
184,S1184,17,Female,4.9,0.0,0.7,No,71.2,6.4,Fair,5,Master,Poor,5,No,96.2
230,S1230,22,Male,4.2,0.3,0.1,No,80.0,7.9,Poor,3,High School,Poor,9,Yes,100.0
315,S1315,24,Female,4.5,3.4,0.4,No,79.6,6.1,Fair,5,High School,Poor,10,No,98.8
320,S1320,24,Female,4.6,1.8,1.5,No,80.9,8.2,Fair,5,NaN,Poor,9,Yes,94.2
336,S1336,21,Female,6.6,2.2,2.7,No,73.0,6.4,Good,2,Bachelor,Poor,8,Yes,100.0
567,S1567,24,Male,4.7,4.1,3.0,No,71.2,4.9,Good,3,Bachelor,Poor,10,Yes,93.1
597,S1597,23,Male,4.9,1.6,4.0,No,87.1,8.0,Fair,5,Bachelor,Poor,7,No,95.5
606,S1606,23,Male,6.8,2.5,0.7,No,97.2,4.4,Good,1,Bachelor,Poor,4,No,93.1
611,S1611,18,Male,6.0,1.6,1.3,No,89.7,5.5,Fair,6,High School,Poor,7,No,100.0
688,S1688,23,Male,5.0,1.8,1.6,No,100.0,5.7,Good,6,High School,Poor,7,No,92.9


In [48]:
# Filter for above average study hours and attendance, but low marks
Filtered = Data[(Data['study_hours_per_day'] >= Data['study_hours_per_day'].mean()) & 
                (Data['attendance_percentage'] >= Data['attendance_percentage'].mean()) & 
                (Data['exam_score'] < 60)]
Filtered

,student_id,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score
179,S1179,19,Female,3.8,1.7,3.8,No,90.2,5.3,Good,2,High School,Average,4,No,58.1
491,S1491,21,Female,4.0,4.3,2.4,No,84.8,8.3,Good,0,High School,Good,2,No,53.0
730,S1730,19,Female,4.3,2.7,2.6,Yes,91.9,5.0,Good,3,Bachelor,Good,1,No,58.4
763,S1763,18,Male,3.9,2.4,0.0,Yes,91.1,5.5,Good,3,High School,Poor,1,Yes,59.5
782,S1782,24,Female,3.9,3.3,1.1,Yes,90.6,8.6,Fair,0,NaN,Good,3,No,56.0
804,S1804,22,Female,4.3,4.0,1.1,Yes,88.4,4.4,Good,2,High School,Average,2,Yes,59.0


**Use as Context**  
The data filtered using pandas used as context to provide LLM the data  
This data retrieval can be quick and effective  


In [49]:
load_dotenv()
client = Groq()

>Specific instructions in terms of Response

In [50]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as CSV formatted string.\
            Answer to the question with details"

In [51]:
# User prompt
Prompt = "Why do you think students score low marks despite attending class and studying well?"

Filtered = Data[(Data['study_hours_per_day'] >= Data['study_hours_per_day'].mean()) & 
                (Data['attendance_percentage'] >= Data['attendance_percentage'].mean()) & 
                (Data['exam_score'] < 60)]
Context = Filtered.to_csv (index=False, float_format='%.1f')

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

**Short answer:**  
Even though the students in the table spend several hours a day studying and have relatively high attendance (mid‑80 % – low‑90 %), many of the other factors that are known to affect learning are sub‑optimal – especially sleep, mental‑health, diet, exercise, and the amount of time they devote to distracting activities (social‑media, Netflix, a part‑time job). Those “hidden” variables can erode the benefits of study time and class attendance and help explain why their exam scores hover in the 50‑60 range rather than climbing higher.

---

## 1. What the data actually show

| Student | Age | Study hrs/day | Attendance % | Sleep hrs | Mental‑health (1 = worst, 5 = best) | Social‑media hrs | Netflix hrs | Part‑time job | Diet quality | Exercise freq (per week) | Exam score |
|---------|-----|---------------|--------------|-----------|--------------------------------------|------------------|-------------|---------------|--------------|--------------------------|--------

>Let's try adapting the response

In [52]:
R_Instr = "Using the context given, provide response to the user question or statement.\
            Context is provided as CSV formatted string.\
            Provide comprehensive response"

In [53]:
# User prompt
Prompt = "Students who have good sleep patterns, are they utlising the time properly?"

# Filter relevant rows for above average sleep time and >75 mark. Only relevant columns
Filtered = Data.loc[(Data['sleep_hours'] >= Data['sleep_hours'].mean()*1.1) & 
                (Data['exam_score'] >= 75),
                ['study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage']]
Context = Filtered.to_csv (index=False, float_format='%.1f')

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

**Short answer:**  
Based on the data you supplied, the students who appear to have “good‑sleep‑friendly” schedules (i.e., low‑intensity screen time, solid attendance and reasonable study hours) are, on the whole, using their waking hours efficiently – they spend most of the day on studying or other productive activities and only a modest amount of time on social‑media/Netflix.  

Below is a step‑by‑step walk‑through of how that conclusion was reached, followed by a few concrete take‑aways and suggestions for anyone who wants to keep the pattern.

---

## 1. How we identified the “good‑sleep” group  

Because the table does **not** contain a direct sleep‑duration column, we used a proxy that is commonly associated with healthy sleep habits:

| Proxy criterion | Rationale |
|-----------------|-----------|
| **Total leisure screen time ≤ 2 h/day** (`social_media_hours + netflix_hours`) | Heavy late‑night screen use is a major disruptor of sleep quality. ≤ 2 h suggests the student likely 

**Get Code from LLM**  
Like in SQL, in pandas also we can take help of LLM to generate code  
This would require specific instructions to be provided for code generation  
The generated code can be used to extract data from dataframe and then use it as context

In [54]:
C_Instr = "For the given pandas dtypes, write code lines that can filter out data to answer user question.\
            Study the pandas dtypes and user question clearly\
            Give only the python code lines for necessary filtering of the Dataframe variable.\
            Code that can filter and return dataframe. Assign results to 'Result'\
            No additional code / string"

In [55]:
# Invoke LLM with prompt and context
Prompt = "Who all have good score despite having attendance below 75?"
# Prompt = "how many scored > 90% with less than 70% attendance?"
# Prompt = "What are the students who score good despite of low study time do?"

Data_Frame_Name = "Data"

Dtype = str(Data.dtypes)

messages=[
    {
        "role": "system",
        "content": C_Instr
    },

    {
        "role": "user",
        "content": "Data.dtype : \n"+ Dtype + "Dataframe variable : 'Data'\n Question : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

Code = completion.choices[0].message.content
Code_Clean = Code.strip ("`")
print (Code_Clean)

exec (Code_Clean)
# print (Result)

Context = Result.to_csv (index=False, float_format='%.1f')

# Invoke LLM with prompt and context
messages=[
    {
        "role": "system",
        "content": R_Instr
    },

    {
        "role": "user",
        "content":"Context : \n"+ Context + "Query : \n" + Prompt
    }
]
completion = client.chat.completions.create(
    messages=messages,
    model="openai/gpt-oss-120b",
)

print (completion.choices[0].message.content)

Result = Data[(Data['attendance_percentage'] < 75) & (Data['exam_score'] >= 75)]
**Students who achieved a high exam score (≥ 85 / 100) while their attendance was **below 75 %**  

| Student ID | Attendance % | Exam Score |
|------------|--------------|------------|
| **S1069** | 72.3 % | 100.0 |
| **S1110** | 64.3 % | 86.5 |
| **S1127** | 71.6 % | 88.0 |
| **S1170** | 74.7 % | 96.6 |
| **S1184** | 71.2 % | 96.2 |
| **S1258** | 66.5 % | 90.8 |
| **S1296** | 74.4 % | 86.2 |
| **S1322** | 68.7 % | 96.1 |
| **S1336** | 73.0 % | 100.0 |
| **S1356** | 70.0 % | 100.0 |
| **S1404** | 74.5 % | 85.6 |
| **S1452** | 71.7 % | 89.3 |
| **S1464** | 73.1 % | 85.3 |
| **S1517** | 73.9 % | 100.0 |
| **S1562** | 63.1 % | 91.7 |
| **S1567** | 71.2 % | 93.1 |
| **S1579** | 69.8 % | 100.0 |
| **S1600** | 74.4 % | 90.2 |
| **S1670** | 72.3 % | 88.9 |
| **S1701** | 74.9 % | 96.4 |
| **S1769** | 64.1 % | 98.8 |
| **S1817** | 71.4 % | 88.2 |
| **S1827** | 71.0 % | 87.5 |
| **S1960** | 69.3 % | 100.0 |

These 

**Pandas SQL**  
using the pandasql library, SQL query can be raised on data frame (treatig like a table)  
The psql library can handle basic SQL queries which are typically handled by SQLite  
Since it can be used as SQL queries, it can be integrated in RAG pipe line (query by LLM etc)

In [56]:
import pandasql as psql

In [57]:
Query = "SELECT * FROM Data WHERE extracurricular_participation = 'Yes'"

result = psql.sqldf(Query)

print(result)

    student_id  age  gender  study_hours_per_day  social_media_hours  \
0        S1000   23  Female                  0.0                 1.2   
1        S1003   23  Female                  1.0                 3.9   
2        S1009   18  Female                  4.8                 3.1   
3        S1016   20    Male                  1.0                 0.6   
4        S1017   24  Female                  3.4                 2.7   
..         ...  ...     ...                  ...                 ...   
313      S1985   18    Male                  5.7                 3.1   
314      S1988   18    Male                  3.3                 2.4   
315      S1995   21  Female                  2.6                 0.5   
316      S1996   17  Female                  2.9                 1.0   
317      S1997   20    Male                  3.0                 2.6   

     netflix_hours part_time_job  attendance_percentage  sleep_hours  \
0              1.1            No                   85.0        